# NB16 — Robust Regression: Handling Outliers

> **StatQuest: "OLS squares residuals — so a single huge outlier can pull the entire line far from the truth."**

---

## The main ideas:

1. OLS squares residuals: a point 10x away from the line gets 100x the penalty
2. That one outlier can completely dominate the OLS fit
3. Robust methods use losses that grow SLOWER than quadratic for large errors
4. **Huber:** quadratic for small errors, linear for large ones
5. **RANSAC:** ignore outliers by finding consensus among majority inliers
6. **Theil-Sen:** use the MEDIAN slope instead of the mean slope


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def flow_diagram(steps, title, colors=None, notes=None, figsize=(14, 2.8)):
    n = len(steps)
    default_colors = ['#1565C0','#2E7D32','#E65100','#6A1B9A',
                      '#00695C','#AD1457','#37474F','#4E342E',
                      '#0277BD','#558B2F','#C62828','#F57F17']
    colors = (colors or default_colors)[:n]
    notes  = notes or ['']*n
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(-0.3, n*3.1); ax.set_ylim(-1.2, 2.4); ax.axis('off')
    bw, bh = 2.6, 1.3
    for i,(step,color,note) in enumerate(zip(steps,colors,notes)):
        x = i*3.1
        box = FancyBboxPatch((x,0.2),bw,bh,boxstyle="round,pad=0.12",
                             facecolor=color,edgecolor='white',linewidth=1.5,alpha=0.90)
        ax.add_patch(box)
        ax.text(x+bw/2,0.2+bh/2,step,ha='center',va='center',fontsize=8.5,
                color='white',fontweight='bold',multialignment='center')
        if note:
            ax.text(x+bw/2,0.02,note,ha='center',va='top',fontsize=7,
                    color='#555',style='italic')
        if i < n-1:
            ax.annotate('',xy=(x+bw+0.38,0.2+bh/2),xytext=(x+bw+0.08,0.2+bh/2),
                       arrowprops=dict(arrowstyle='->',color='#444',lw=2.2))
    ax.set_title(title,fontsize=11,fontweight='bold',pad=6,color='#222')
    plt.tight_layout(pad=0.4); plt.show()

flow_diagram(
    steps=[
        'Data with\noutliers',
        'OLS: squares\nall residuals\n(outlier dominates)',
        'Huber: quadratic\nfor small e,\nlinear for large',
        'RANSAC:\nrandom consensus\ncount inliers',
        'Theil-Sen:\nmedian of all\npairwise slopes',
        'Compare all\nthree vs\nOLS',
        'Choose method\nbased on\ncontamination %',
    ],
    title='NB16 Conceptual Flow: Robust Regression Methods',
    colors=['#C62828','#37474F','#1565C0','#2E7D32','#E65100','#6A1B9A','#00695C'],
    figsize=(16, 2.8),
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import (LinearRegression, HuberRegressor,
                                  RANSACRegressor, TheilSenRegressor)

np.random.seed(42)
n = 60
X_clean = np.linspace(0, 10, n).reshape(-1,1)
y_clean = 2*X_clean.ravel() + 3 + np.random.normal(0, 1.5, n)

# Add 8 gross outliers
X_out = np.array([[1],[3],[5],[9],[9.5],[2],[8],[7]])
y_out = np.array([22, 26, 28, -5, -8, 24, -3, -4])
X_all = np.vstack([X_clean, X_out])
y_all = np.concatenate([y_clean, y_out])

models = {
    'OLS':       LinearRegression(),
    'Huber':     HuberRegressor(epsilon=1.35),
    'RANSAC':    RANSACRegressor(random_state=42),
    'Theil-Sen': TheilSenRegressor(random_state=42),
}
colors_m = ['red','green','orange','purple']

X_plot = np.linspace(-1, 11, 200).reshape(-1,1)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_clean, y_clean, s=25, color='steelblue', alpha=0.7, zorder=3, label='Clean data')
ax.scatter(X_out, y_out, s=100, color='black', marker='x', linewidths=2.5, zorder=4, label='Outliers')

for (name, m), col in zip(models.items(), colors_m):
    m.fit(X_all, y_all)
    if hasattr(m, 'estimator_'):
        pred = m.estimator_.predict(X_plot)
    else:
        pred = m.predict(X_plot)
    ax.plot(X_plot, pred, color=col, linewidth=2.5, label=name)

ax.set_xlabel('X'); ax.set_ylabel('y')
ax.set_title('Robust methods resist outliers; OLS is pulled away from the truth')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("True slope = 2.0")
for (name, m), col in zip(models.items(), colors_m):
    slope = m.estimator_.coef_[0] if hasattr(m,'estimator_') else m.coef_[0]
    print(f"  {name:<12}: estimated slope = {slope:.4f}")


## Huber loss explained

```
L_Huber(r) =  r^2 / 2              if |r| <= epsilon
              epsilon * (|r| - epsilon/2)   if |r| > epsilon
```

- For small residuals (|r| < epsilon): behaves exactly like OLS (quadratic)
- For large residuals (outliers): switches to LINEAR growth — outlier is penalised linearly, not quadratically
- epsilon = 1.35 gives 95% efficiency on normally distributed data

**Theil-Sen:** slope = median of ALL pairwise slopes between every pair of points.
Can handle up to 29.3% contamination.

**RANSAC:** randomly sample minimal subsets, fit, count inliers, keep best model.


## When to use which

| Method | Breakdown point | Speed | Use when |
|--------|----------------|-------|----------|
| OLS | 0% (one outlier can ruin it) | Fastest | Data is clean |
| Huber | ~20% | Fast | Moderate contamination |
| RANSAC | >30% | Moderate | Known to have gross outliers |
| Theil-Sen | 29.3% | Slow O(n^2) | Small n, high contamination |

**Next: NB17 — Quantile regression (model the median, not the mean).**
